# NE & ACh experiment planning (VISp vs V2M)

Uses **existing synthesis outputs** (notebook 05) — no re-run of 01–04 required.

Produces:
- Expression heatmaps (coarse L2/3 IT, L5 IT, L5 ET × VISp/V2M)
- Pharmacology priority ranking → **agonist/antagonist ordering list**
- Per-receptor intrinsic excitability table (CSV + text)
- Combined **low vs high NE/ACh** scenario predictions (affinity-weighted)
- `EXPERIMENT_PLAN_NE_ACh.txt` for meetings and lab ordering

Set `CELL_TYPE_LEVEL = "supertype"` to use the June supertype synthesis run.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import load_config, resolve_output_dir, start_run
from src.experiment_planning import (
    find_latest_synthesis_run,
    run_experiment_planning,
)
from src.utils import print_path

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "query_config.yaml"
config = load_config(CONFIG_PATH)
EXPLORATION_ROOT = resolve_output_dir(cfg=config)

# Override config level for this notebook (supertype figures requested)
CELL_TYPE_LEVEL = "supertype"  # or "subclass" to match query_config.yaml
config["cell_type_level"] = CELL_TYPE_LEVEL

SYNTHESIS_RUN = find_latest_synthesis_run(EXPLORATION_ROOT, CELL_TYPE_LEVEL)
print_path("Synthesis evidence:", SYNTHESIS_RUN)

OUTPUT_DIR = start_run(
    PROJECT_ROOT,
    config,
    dataset="experiment_planning",
    exploration_root=EXPLORATION_ROOT,
    notebook="06_experiment_planning",
)
print_path("Output:", OUTPUT_DIR)

In [ ]:
result = run_experiment_planning(
    config,
    synthesis_run=SYNTHESIS_RUN,
    output_dir=OUTPUT_DIR,
    cell_type_level=CELL_TYPE_LEVEL,
)

summary = result["summary"]
order_table = result["order_table"]
scenario_table = result["scenario_table"]

print(f"Coarse summary rows: {len(summary):,}")
print(f"Pharmacology order rows: {len(order_table):,}")
print(f"Transmitter scenarios: {len(scenario_table):,}")
print()
for key, path in sorted(result["paths"].items()):
    print_path(key, path)

In [ ]:
# Top ordering priorities (high tier first)
display(
    order_table[
        ["coarse_type", "brain_area", "gene", "receptor", "coupling",
         "confidence_tier", "expression", "priority_score",
         "agonists", "antagonists"]
    ].head(20)
)

In [ ]:
# Combined transmitter scenarios (low vs high NE / ACh)
display(
    scenario_table[
        ["transmitter", "coarse_type", "brain_area", "concentration",
         "active_genes", "coupling_Gi", "coupling_Gq", "coupling_Gs", "coupling_ionotropic",
         "axis_excitability", "axis_rin", "axis_ih", "axis_mcurrent", "axis_adaptation"]
    ]
)

## Text plan

Open `EXPERIMENT_PLAN_NE_ACh.txt` in the run folder for the full narrative, compound list, and ephys readout suggestions.

In [ ]:
plan_path = OUTPUT_DIR / "EXPERIMENT_PLAN_NE_ACh.txt"
print(plan_path.read_text(encoding="utf-8")[:4000])